# XGBoost GBDT on the ArchPower Event Dataset

User interface for the GBDT model. All heavy lifting lives in `src/event_gbdt.py`; this notebook just configures, calls, and displays results.

Run from the `script/` directory. Outputs land in `output/gbdt_<ARCH>/` at the repo root.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join("..", "src"))

import numpy as np
import matplotlib.pyplot as plt

from event_utils import COMP_NAMES, prepare_data_numpy, compute_metrics
from event_attention import EVENT_NAMES
import event_gbdt as gbdt

In [ ]:
ARCH = "BOOM"
VAL_RATIO = 0.2
SEED = 42
N_ESTIMATORS = 100
MAX_DEPTH = 6
LEARNING_RATE = 0.3

# Tree visualization: which component's model and which tree to render
VIZ_COMP = "Total"
VIZ_TREE = 0

OUT_DIR = os.path.join("..", "output", f"gbdt_{ARCH}")
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Load event dataset

In [ ]:
data = prepare_data_numpy(ARCH, val_ratio=VAL_RATIO, seed=SEED)
n_train = data["X_train"].shape[0]
n_val = data["X_val"].shape[0]
n_targets = data["y_train"].shape[1]
print(f"train: {n_train}  val: {n_val}  features: {data['X_train'].shape[1]}  targets: {n_targets}")

## 2. Train 12 XGBRegressors (one per power component)

In [ ]:
models = gbdt.train_models(
    data,
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    feature_names=EVENT_NAMES,
)

## 3. Evaluate on validation and training sets

In [ ]:
train_pred, val_pred = gbdt.predict_both(models, data)
train_true = data["y_train_raw"]
val_true = data["y_val_raw"]
val_metrics = compute_metrics(val_pred, val_true)

print(f"\n  {'Component':12s} {'RMSE':>10s} {'MAPE%':>8s} {'R2':>8s}")
for i, name in enumerate(COMP_NAMES):
    print(f"  {name:12s} {val_metrics['rmse'][i]:10.6f} "
          f"{val_metrics['mape'][i]:8.2f}% {val_metrics['r2'][i]:8.4f}")

## 4. Predicted vs True scatter (per component, train + test)

Each subplot has true on the y-axis, predicted on the x-axis. Blue = train, orange = test. Dashed line is y = x (perfect prediction).

In [ ]:
gbdt.plot_pred_vs_true(train_true, train_pred, val_true, val_pred,
                       val_metrics, OUT_DIR)
plt.show()

## 5. Feature importance -- gain / cover / weight

XGBoost's three importance types (the regression analog of sklearn's gini-based feature importance):

- **gain**: average loss reduction contributed by splits on this feature
- **cover**: average number of samples affected by splits on this feature
- **weight**: number of times this feature is used to split

In [ ]:
imp = gbdt.compute_feature_importance(models, EVENT_NAMES)
gbdt.plot_per_component_importance(imp, EVENT_NAMES, OUT_DIR, top_k=15)
gbdt.plot_global_importance(imp, EVENT_NAMES, OUT_DIR, top_k=20)
plt.show()

# Top-10 table printout
imp_global = imp.mean(axis=0)
print("\nTop-10 features by global mean importance")
print(f"{'rank':>4} | {'gain':<35} | {'cover':<35} | {'weight':<35}")
print("-" * 120)
for r in range(10):
    row = []
    for ti in range(3):
        v = imp_global[:, ti]
        idx = np.argsort(v)[::-1][r]
        row.append(f"{EVENT_NAMES[idx]} ({v[idx]:.3f})")
    print(f"{r+1:>4} | {row[0]:<35} | {row[1]:<35} | {row[2]:<35}")

## 6. Tree visualization

Render a single tree from the chosen component model. Each internal node shows `feature < threshold`; each leaf shows the leaf value. Adjust `VIZ_COMP` / `VIZ_TREE` in the config cell to inspect a different tree.

If the system `graphviz` binary (`dot`) is not installed, the helper falls back to a text dump (install with `brew install graphviz` on macOS to get the graphical version).

In [ ]:
gbdt.plot_tree(models[COMP_NAMES.index(VIZ_COMP)], VIZ_COMP, VIZ_TREE, OUT_DIR)
plt.show()

## 7. SHAP attribution analysis

Computes SHAP values on the validation set with `shap.TreeExplainer`. Each component model gets two plots: a beeswarm summary (top-20 features) and a mean(|SHAP|) bar chart.

The global plot at the bottom averages |SHAP| across all 12 component models.

In [ ]:
shap_values = gbdt.compute_shap_values(models, data["X_val"])
gbdt.plot_shap_per_component(shap_values, data["X_val_raw"], EVENT_NAMES, OUT_DIR)
gbdt.plot_global_shap(shap_values, EVENT_NAMES, OUT_DIR, top_k=20)
plt.show()

## 8. Generate markdown report

Writes the same `output/gbdt_<ARCH>/report.md` that `src/event_gbdt.py` produces from the CLI.

In [ ]:
gbdt.write_report(
    arch=ARCH,
    data=data,
    metrics=val_metrics,
    val_ratio=VAL_RATIO,
    hyperparams={
        "n_estimators": N_ESTIMATORS,
        "max_depth": MAX_DEPTH,
        "learning_rate": LEARNING_RATE,
    },
    out_path=os.path.join(OUT_DIR, "report.md"),
)
print(f"\nDone! All outputs in: {os.path.abspath(OUT_DIR)}/")